In [ ]:
# Use only the package used here
import math

In [ ]:
# use for readDataFromFile and ReadDEGFromFile
def replace(text):
  characters = "\n "
  return ''.join( x for x in text if x not in characters)

In [ ]:
def readDataFromFile(filename):
  sampleCls = {}
  exp = {}
  #Problem 1. precessing your train or test file
  #_____________________________________
  #your code
  #sampleCls -- A dictionary which map a sample to class(non-tumor : True, cancer : False) ex){1:True ....}
  #exp -- A dictionary which map a class to expression data ex) {1:{gene1 : 0.784 ...}, ....}
  #_____________________________________
  with open(filename) as file:
    sample_class = file.readline().split()
    for i, class_label in enumerate(sample_class[1:], start = 1):
      sampleCls[i] = class_label == 'cancer'
      exp[i] = {}
    
    for line in file:
      value = line.split()
      for i, expression_value in enumerate(value[1:], start = 1):
        exp[i][value[0]] = float(expression_value)

  return sampleCls, exp

In [ ]:
cnt = 0
filename = "Lab10_GSE13164_test.txt"# your filename
sampleCls, exp = readDataFromFile(filename)
print("TEST1: print sampleCls")
for k,v in sampleCls.items():
  print("Sample", k, ": ", "Normal" if v else "Disease")
  cnt+=1
  if cnt > 20:
    print("...")
    break
print("TEST2: print exp")

cnt = 0
for k,v in exp.items():
  for k2, v2 in v.items():
    print("(sample", k, ",", k2,"): ",v2)
    cnt+=1
    if cnt > 20:
      print("...")
      break

In [ ]:
def readDEGFromFile(filename):
  deg = []
  #Problem 2. precessing your DEG file 
  #_____________________________________
  #your code
  #deg -- A list which includes DEGs ex) [DEG1, DEG2, ... ]
  #_____________________________________
  with open(filename) as file:
    next(file)
    deg = [line.split()[0] for line in file]

  return deg

In [ ]:
#Printing out all of the name of DEGs
print("TEST3: print DEG list")
filename = "Lab10_GSE13164_test.txt"# your filename
cnt = 0
degs = readDEGFromFile(filename)
for deg in degs:
  print(deg)
  cnt += 1
  if cnt > 20:
    print("...")
    break

In [ ]:
def findCutoffValue(gene, sample_lists, sampleCls):
  SuM_norm = 0 
  SuM_cancer = 0 
  cnt_n = 0
  cnt_c = 0
  # Problem 3. Write your own code in here.Please refer "Lab10_PrelabGuideline.pdf"
  # Any of your own idea for finding classifying value is okay (additional score could be given).
  # Pay attention to treat when all classes of samples are same.
  #_____________________________________
  # your code
  #_____________________________________
  for i in range(len(sample_lists)):
    sample = sample_lists[i]
    if sampleCls[sample]:
      SuM_norm += sample
      cnt_n += 1
    else:
      SuM_cancer += sample
      cnt_c += 1
    
    if cnt_n == 0 or cnt_c == 0: # When all samples have the same class 
      return None

    mean_norm = SuM_norm / cnt_n
    mean_cancer = SuM_cancer / cnt_c

    cutoffvalue = (mean_norm + mean_cancer) / 2

  return cutoffvalue

In [ ]:
sample_list = [1,2,4,5,6,3]
gene = "211644_x_at"
cnt = 0
print("TEST4: print class of samples in 'sampleList'")
for sample in sample_list:
  print("sample#: ", sample,end='\t')
  print("sample# in map: ",sample, end = '\t')
  print("sample class in map: ","Normal" if sampleCls[sample] else "Disease")

print("TEST5: find expression value for gene of given sample")
print("input gene: ",gene)
for sample in sample_list:
  print("Expression value of ",gene,"sample_list: ", end='')
  print(exp[sample][gene])

print("TEST6: Find cutoff value for a gene of given samples")
print("input gene: ", gene)
print("cutoffvalue : ", findCutoffValue(gene, sample_list, sampleCls))

In [ ]:
def entropycal(fOrM,lAtT):
  if fOrM == 0 and lAtT == 0:
    return 0.00
  elif fOrM == 0:
    return 0.00
  elif lAtT == 0:
    return 0.00
  else:
    p = float(fOrM / (fOrM + lAtT))
    return -p*math.log(p, 2) - (1-p)*math.log(1 - p, 2) #____________________ Problem4. Fill in the blank. Think how to calcuation of entropy

In [ ]:
def getInfoGain(gene, cutoffValue, sample_lists):
  Norm, Cancer, Norm_h, Norm_l, Cancer_h, Cancer_l = 0,0,0,0,0,0
  for sample in sample_lists:
    if sampleCls[sample]:
      Norm+=1
      if cutoffValue is not None and exp[sample][gene] >= cutoffValue: Norm_h+=1 #ensure that cutoffValue is not 'None'
      else: Norm_l+=1
    elif not sampleCls[sample]:
      Cancer+=1
      if cutoffValue is not None and exp[sample][gene] >= cutoffValue: Cancer_h+=1 #ensure that cutoffValue is not 'None'
      else: Cancer_l+=1

  entro_1 = entropycal(Norm, Cancer)
  entro_h = entropycal(Norm_h, Cancer_h)
  entro_l = entropycal(Norm_l, Cancer_l)

  output = entro_1 - entro_h*(Norm_h + Cancer_h)/(Norm + Cancer) - entro_l*(Norm_l + Cancer_l)/(Norm + Cancer) #____________________ Problem5. Fill in the blank. Think how to calculate information gain.
  return output

In [ ]:
sample_list = [1,2,4,5,6,3]
gene = "211644_x_at"
cutoffValueList = [findCutoffValue(gene, sample_list, sampleCls), 3.1,4.4]

print("TEST7: Check information gain value for a gene of given samples with various cutoff value")
print("input gene: ", gene)
for cutoffValue in cutoffValueList:
  print("InfoGain of ",gene, "for samples with cutoff",cutoffValue,":", getInfoGain(gene,cutoffValue,sample_list))